# 10 · GPT-6 Astra as a geospatial analysis partner


**Mission (40 minutes):** prepare an evidence packet, construct a structured Responses API request, inspect a labeled illustrative response, and reject unsupported citations. The notebook runs without credentials. Optional live use happens in a separate local Python process with a server-side key, never in JupyterLite or GitHub Pages.

The documented model supports text/image input, reasoning, tool calling and structured outputs. The workflows below apply these general capabilities to maps; they are not evidence of a dedicated GIS engine or an accuracy guarantee. Distances, rates, and spatial tests stay in explicit code.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

In [ ]:
from astra_contract import build_request, validate_briefing, example_briefing
import json
bundle=evidence_bundle()
request=build_request(bundle)
print('Model:',request['model'])
print('Aggregate reported cases:',bundle['metrics']['reported_cases'])
print('Evidence IDs:',[s['id'] for s in bundle['sources']])
download('evidence-bundle.json',bundle)
download('astra-request.json',request)

## 1. Give Astra a bounded job
Ask for an exercise situation report, with every finding tied to a supplied source ID, explicit unknowns, and verification steps. Data text is evidence, not instructions. The request disables storage (`store: false`) and exposes no action tools. That setting is not a promise about all service retention; consult your deployment's data controls.

**Useful extensions:** critique a map legend; explain a denominator mismatch; generate a test for coordinate order; compare a map screenshot with its manifest; summarize authorized source documents; propose sensitivity analyses; draft a multilingual briefing for human review. No credentials are needed to study the request structure.

In [ ]:
print(request['instructions'])
print(json.dumps(request['text']['format'],indent=2))

## 2. Validate an illustrative response
This deterministic fixture is **not an API response**. It demonstrates the schema and citation checks. Schema conformance is not factual validation; a reviewer must check each claim against the numerical packet and source provenance.

In [ ]:
brief=example_briefing(bundle)
validate_briefing(brief,bundle)
display(Markdown('### ILLUSTRATIVE FIXTURE — NOT MODEL OUTPUT'))
for finding in brief['findings']:
    print(finding['claim'],'Evidence:',', '.join(finding['evidence_ids']))
bad=json.loads(json.dumps(brief));bad['findings'][0]['evidence_ids']=['S999']
try:
    validate_briefing(bad,bundle)
except ValueError as error:
    print('Expected rejection:',error)

## 3. Optional live run, outside the browser kernel
Download `evidence-bundle.json`. In a local checkout, run the following in a **terminal**, not a JupyterLite cell:

```bash
python tools/astra_brief.py --input evidence-bundle.json --output request.json
# Above: dry-run request only, no API call.
# Set OPENAI_API_KEY in the local process environment, then deliberately opt in:
python tools/astra_brief.py --input evidence-bundle.json --output briefing.json --live
```

For a map image you are authorized to send, add `--image map.png` to either command. The image is paired with the evidence packet, not used as a coordinate measurement system. `--live` incurs API usage and requires account access to `gpt-6-astra`. The CLI rejects invalid/unsupported response structures and unknown source IDs; refusals and incomplete responses are surfaced as errors. No live API call is part of CI.

To review the result in JupyterLite, upload `briefing.json` beside this notebook, load it with `json.loads(Path('briefing.json').read_text())`, and call `validate_briefing(result, bundle)`.

## Evaluation lab
Score a real response 0/1 on each: correct total, correct date window, known source IDs, reporting-delay caveat, no causal claim, no invented coordinates, clear next verification step. All seven are required before using the text in an **exercise** brief. Try a malicious instruction in a source description and confirm it does not change the task or lead to a fabricated citation.

**Sources verified 2 October 2026:** [GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [images and vision](https://developers.openai.com/api/docs/guides/images-vision), [API data controls](https://developers.openai.com/api/docs/guides/your-data). Account availability and provider behavior may change.